# IndicEAR design A: FLEURS ablation and robustness probe (historical record)

Frozen EmbeddingGemma 2 audio encoder (`914f7f89`), never modified. Layers 1-6 are cached once; layers 7-12 run every step with trainable adapters. FLEURS train/dev for 12 languages; **FLEURS test is never read**.

| Variant | What it adds |
| --- | --- |
| v1 | linear CTC on frozen layer 12 (the probe baseline) |
| v2 | adapters on layers 7-12 + final CTC |
| v3 | + self-conditioned phonetic CTC at layers 6 and 9 |
| v4 | + two-level script output p(script) x p(subword given script), CTC form |
| v5 | + CR-CTC (optional) |
| v6 | + factored RNN-T head with a CTC branch at weight 0.3 (design A with RNN-T standing in for TDT; optional) |

This is the first ablation (23 October 2026) that chose variant v4; the released model's recipe is `train_colab.ipynb`.

**How to run.** Every long step is a bash script started with `scripts/launch.sh` under `nohup`, so it survives notebook and SSH disconnects. The same commands work over SSH (see README.md). Every step is resumable: rerunning a cell continues where it stopped, and checkpoints are synced to `$SYNC` (Google Drive) after each save.

**Runtime:** an A100 (40/80 GB), an L4 (24 GB) or an RTX PRO 6000 (96 GB). Batch budgets are sized from free VRAM at startup, so no edits are needed.

**Checkpoints can be pulled to a workstation.** Each run directory (`$ROOT/runs/<variant>/`) holds `LATEST.json`, a manifest of complete files with sha256. On the workstation, `scripts/sync_runs.sh <session> /content/indicear/runs/<variant>` pulls only those files over one multiplexed SSH connection, verifies them and keeps the last 2 checkpoints plus the best in `~/indicear-runs/<variant>/`. If the VM is lost: `scripts/resume_on.sh <new session> <variant>` uploads the verified copy and reruns the run's `command.sh`. Google Drive (`$SYNC`) is only an optional second copy.

In [ ]:
# @title 1. GPU, CPU, RAM and disk
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
# @title 2. Code and paths
!git clone -q https://github.com/rahulsharmavishwakarma/indicear /content/indicear-code
%env CODE=/content/indicear-code
%env ROOT=/content/indicear
%env VENV=/content/venv314t
# Optional second copy on Drive (mount it first, cell 3).
%env SYNC=/content/drive/MyDrive/indicear
!ls $CODE/scripts

In [ ]:
# @title 3. Mount Google Drive (persistent checkpoints) and the Hugging Face token
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # FLEURS is ungated; the token avoids rate limits

In [ ]:
# @title 4. Free-threaded Python 3.14t environment (uv, exact pins)
!bash $CODE/scripts/setup_env.sh

In [ ]:
# @title 5. Optional: MUSAN noise (about 11 GB download, only noise/ kept)
# Without it the robustness probe uses synthetic colored noise, and the reports say so.
!bash $CODE/scripts/launch.sh musan $CODE/scripts/musan_noise.sh
%env NOISE_DIR=/content/indicear/musan/musan/noise

In [ ]:
# @title 6. Prepare data: FLEURS manifests, tokenizer, code-switch dev, negatives, label rates (background)
!bash $CODE/scripts/launch.sh prep $CODE/scripts/prep.sh

In [ ]:
# @title Progress of prep (rerun any time)
!tail -n 20 $ROOT/logs/prep.log

In [ ]:
# @title 7. Cache layers 1-6 once (background; resumable per part, df checked first)
# AUG_VIEWS=2 adds two augmented copies per training utterance (speed, reverb + point noise,
# codecs, babble, noise): about 3x the cache size.
!WAIT_FOR=$ROOT/prep_report.json AUG_VIEWS=0 STORAGE=bf16 bash $CODE/scripts/launch.sh cache $CODE/scripts/cache.sh

In [ ]:
# @title Progress of the cache (throughput in hours of audio per hour, GPU utilization)
!tail -n 8 $ROOT/logs/cache.log; tail -n 3 $ROOT/cache/layer6/gpu.csv

In [ ]:
# @title 8. Train v1, the frozen-encoder baseline (background)
!WAIT_FOR=$ROOT/cache/layer6/cache.done VARIANTS=v1 PARALLEL=1 bash $CODE/scripts/launch.sh ablation-v1 $CODE/scripts/ablation.sh

In [ ]:
# @title 9. Robustness probe of the frozen encoder with the v1 heads (background, 3 jobs on one GPU)
# Clean, SNR 20/10/5 (noise and babble), reverberation, 8 kHz telephone, AMR-NB/GSM/Opus, speed 0.9/1.1.
!WAIT_FOR=$ROOT/results/v1.json PARALLEL=3 bash $CODE/scripts/launch.sh robustness $CODE/scripts/robustness.sh

In [ ]:
# @title 10. Train v2-v4 (background). PARALLEL=3 shares the GPU between three variants.
# Add v5 v6 to VARIANTS for CR-CTC and the RNN-T head.
# Queued behind the robustness probe so the jobs do not compete for VRAM.
!WAIT_FOR=$ROOT/results/robustness/v1/summary.json VARIANTS="v2 v3 v4" PARALLEL=3 bash $CODE/scripts/launch.sh ablation $CODE/scripts/ablation.sh

In [ ]:
# @title Status of every job (rerun any time)
!bash $CODE/scripts/status.sh

In [ ]:
# @title 11. Results
import json, pathlib
root = pathlib.Path('/content/indicear/results')
for name in ['summary.md', 'robustness/v1/summary.md']:
    path = root / name
    print(f'== {name}'); print(path.read_text() if path.exists() else 'not yet')
print(json.loads((pathlib.Path('/content/indicear') / 'prep_report.json').read_text())['label_rates'])

In [ ]:
# @title 12. Copy results to Drive, then release the runtime
!mkdir -p $SYNC/results && cp -r $ROOT/results/. $SYNC/results/ && ls $SYNC/results
print('Results copied. Release the runtime now: Runtime > Disconnect and delete runtime.')